# Part 2 — When rebuilds stop scaling

Part 1 ended with a working pipeline and one unexplained line. Every model in
it is a complete statement of its table, rebuilt from scratch each run
except one table: `fct_orders`. The model config includes `materialized='incremental'`, which part
1 walked straight past.

Thirty orders rebuild in about three seconds. Thirty million do not, which is
why that line is there. dbt's own
[materialization guidance](https://docs.getdbt.com/best-practices/materializations/1-guide-overview)
is a ladder: view, then table, then incremental only once the build itself
is too slow or too expensive. This is the last rung, reached deliberately.

Writing on
[the limits of incrementality](https://discourse.getdbt.com/t/on-the-limits-of-incrementality/303),
dbt's founder called incrementality "*hard*... easy to screw up" and "fairly
hard to test", and the thread remains one of the most-read posts on dbt's own
forum.

**What you'll see**

- what an incremental model actually does on the second run
- three ways to get it wrong, each broken live and then fixed
- why the choice of *which timestamp to filter on* is the one that quietly
  loses money

## Setup

Reset, then build the project from part 1.

In [1]:
!reset

... restoring raw seed data and dropping dbt schemas


✓ lab restored to its starting state


In [2]:
import pandas as pd

from lab import artifacts, scratch, shop, verdict
from lab.runner import dbt, show

result = dbt("run", "--select", "staging +fct_daily_revenue", quiet=True)
assert result.success
assert shop.scalar("SELECT COUNT(*) FROM marts.fct_orders") == 30

verdicts = []  # each scenario appends a row; the closing table collects them
print("✓ baseline: 30 orders in the mart")

✓ baseline: 30 orders in the mart


## What incremental means

Here is `fct_orders`. Part 1 skipped past its config block; read it now.

In [3]:
show("models/marts/fct_orders.sql")

-- The order fact. Incremental: once this table is too big to rebuild hourly,
-- each run appends or updates only what changed since the last one.
--
-- The three config choices below are not obvious, and none of them is the
-- default. Notebook 02 is about why each one is what it is.
{{ config(
    materialized='incremental',
    incremental_strategy='merge',
    unique_key='order_id',
    on_schema_change='fail'
) }}

select
    order_id,
    customer_id,
    product_id,
    sku,
    category,
    status,
    quantity,
    unit_price,
    line_total,
    placed_at,
    updated_at,
    loaded_at
from {{ ref('int_orders__enriched') }}

{% if is_incremental() %}
where loaded_at > (select coalesce(max(loaded_at), '1900-01-01') from {{ this }})
{% endif %}



Each config setting is a decision.

`materialized='incremental'` tells dbt: on the first run, build the whole
table. On every subsequent run, do not drop it. Work out what is new and
add only that.

`incremental_strategy='merge'` with `unique_key='order_id'` says what to do
with a row whose key is already present: update it in place rather than insert
a second copy.

`on_schema_change='fail'` decides what happens when the shape of the incoming
data changes. Scenario 4 comes back to it.

The `{% if is_incremental() %}` block is how you say what "new" means. It is
skipped on the first run and included on every later one, so the `where`
clause inside it filters the source down to rows past the high-water mark
already in the table — the *watermark*, from here on. (`{{ this }}` in that
clause is the table being built.)

---

## Scenario 1 · Run it again

The simplest thing that can happen: nothing changed upstream. Re-run the
model.

In [4]:
result = dbt("run", "--select", "fct_orders", quiet=True)
assert result.success

In [5]:
merged = artifacts.run_results()["model.shop.fct_orders"]
assert merged["status"] == "success" and merged["rows"] == 0
assert shop.scalar("SELECT COUNT(*) FROM marts.fct_orders") == 30
verdicts.append(("1 · re-run, nothing new", "merge + unique_key (committed)", "graceful", "—",
                 "0 rows merged, count unchanged"))
print("✓ 0 rows merged · still 30 orders")

✓ 0 rows merged · still 30 orders


Zero rows touched, and the table is unchanged. That is the property everything
else depends on: schedulers retry, people re-run things by hand, and none of
that is allowed to corrupt the mart.

---

## Scenario 2 · The order that arrived late

Now the interesting one.

An order is placed on 20 July. Its row does not reach the warehouse until the
30th: an outage, a stuck queue, a partner feed catching up. Late data always
has a story.

Two timestamps now disagree, and the incremental filter has to pick one:

![the two clocks](img/two-clocks.svg)

The intuitive choice is event time: *"give me orders newer than the newest
order I have."*

We shall compare two incremental models that are based on different watermark dates:

1. **event time**: when the order was placed
2. **arrival time**: when the row reached the warehouse

In [6]:
scratch.model("fct_orders_event_time", """
{{ config(materialized='incremental', incremental_strategy='merge',
          unique_key='order_id', schema='scratch') }}

select order_id, customer_id, status, quantity, line_total, placed_at, loaded_at
from {{ ref('int_orders__enriched') }}

{% if is_incremental() %}
-- watermark on EVENT time: "orders placed after my newest placed_at"
where placed_at > (select coalesce(max(placed_at), '1900-01-01') from {{ this }})
{% endif %}
""")

result = dbt("run", "--select", "fct_orders_event_time", quiet=True)
assert result.success
assert shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_event_time") == 30
print("✓ naive variant built · 30 rows, identical to the mart so far")

✓ naive variant built · 30 rows, identical to the mart so far


Both models now hold the same 30 orders. Land the late one:

In [7]:
shop.arrive_late(
    order_id=31, customer_id=4, product_id=4, quantity=50, unit_price_cents=2200,
    placed_at="2026-07-20T11:20:00", loaded_at="2026-07-30T02:05:00",
)
shop.df("SELECT order_id, placed_at, _loaded_at FROM raw.orders WHERE order_id = 31")

,order_id,placed_at,_loaded_at
0,31,2026-07-20 11:20:00,2026-07-30 02:05:00


Fifty hi-vis vests: $1,100 that the 20th of July earned and nobody has
counted. Run both models, unchanged:

In [8]:
result = dbt("run", "--select", "fct_orders_event_time fct_orders fct_daily_revenue", quiet=True)
assert result.success

pd.DataFrame({
    "model": ["event-time watermark", "arrival-time watermark (committed)"],
    "rows": [shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_event_time"),
             shop.scalar("SELECT COUNT(*) FROM marts.fct_orders")],
    "has order #31": [
        bool(shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_event_time WHERE order_id = 31")),
        bool(shop.scalar("SELECT COUNT(*) FROM marts.fct_orders WHERE order_id = 31")),
    ],
})

,model,rows,has order #31
0,event-time watermark,30,False
1,arrival-time watermark (committed),31,True


### Verdict: the event-time watermark lost the sale, and said nothing

Order #31 was placed on the 20th. The naive model's high-water mark is already
at the 28th. The row is *behind* the mark, so its filter discarded it. Because
an event's timestamp never changes, it will be discarded on every
future run too. That row is gone permanently.

The committed model filters on `loaded_at`, the loader's own clock. Anything
that arrives after the last run gets processed, however old the event it
describes.

Here is what that costs downstream, on the day itself:

In [9]:
revenue_jul20 = shop.scalar(
    "SELECT revenue FROM marts.fct_daily_revenue WHERE order_date = '2026-07-20'")
assert shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_event_time WHERE order_id = 31") == 0
assert shop.scalar("SELECT COUNT(*) FROM marts.fct_orders WHERE order_id = 31") == 1
assert revenue_jul20 == 1320.00  # $220 on time + $1,100 late
verdicts.append(("2 · order arrives late", "watermark on event time", "silent", "yes",
                 "row is permanently below the mark"))
verdicts.append(("2 · order arrives late", "watermark on arrival time (committed)", "graceful", "—",
                 "caught on the next run; revenue corrected"))
print(f"✓ committed mart caught it — 20 July revenue = ${revenue_jul20}")

✓ committed mart caught it — 20 July revenue = $1320.00


**One honest caveat.** Arrival time only works while the loader stamps
honestly. A loader that backfills history without bumping `_loaded_at`
reopens the same hole one layer down. The usual insurance is a lookback
window, reprocessing a trailing margin on every run:

```sql
where loaded_at > dateadd(day, -3, (select max(loaded_at) from {{ this }}))
```

That deliberately reprocesses rows you already have, which is only safe
because of the next scenario.

---

## Scenario 3 · The row that changed

Order #12 has sat at `placed` since 10 July. On the 30th it ships. The source
updates the row in place and the loader re-lands it with a fresh `_loaded_at`,
so it crosses the watermark again.

What happens now is entirely down to `incremental_strategy`. The simplest
strategy is `append` ("new rows get inserted"), which is also most people's
default mental model. Build that variant:

In [10]:
scratch.model("fct_orders_append", """
{{ config(materialized='incremental', incremental_strategy='append', schema='scratch') }}

select order_id, customer_id, status, quantity, line_total, placed_at, loaded_at
from {{ ref('int_orders__enriched') }}

{% if is_incremental() %}
where loaded_at > (select coalesce(max(loaded_at), '1900-01-01') from {{ this }})
{% endif %}
""")

result = dbt("run", "--select", "fct_orders_append", quiet=True)
assert result.success
assert shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_append") == 31
print("✓ append variant built: 31 rows, same as the mart")

✓ append variant built: 31 rows, same as the mart


In [11]:
shop.update_order_status(12, "shipped", updated_at="2026-07-30T08:00:00")
result = dbt("run", "--select", "fct_orders_append fct_orders", quiet=True)
assert result.success

pd.concat([
    shop.df("SELECT 'append variant' AS model, order_id, status, loaded_at "
            "FROM scratch.fct_orders_append WHERE order_id = 12"),
    shop.df("SELECT 'merge (committed)' AS model, order_id, status, loaded_at "
            "FROM marts.fct_orders WHERE order_id = 12"),
])

,model,order_id,status,loaded_at
0,append variant,12,placed,2026-07-10 12:05:00
1,append variant,12,shipped,2026-07-30 08:00:00
0,merge (committed),12,shipped,2026-07-30 08:00:00


### Verdict: append turned one order into two

The updated row crossed the watermark (correctly: it *is* new data) and
`append` did the only thing it knows how to do. The variant now reports 32
orders and holds order #12 twice: once placed, once shipped. Every count, sum
and join over that table is now wrong, and the run that did it finished green.

`merge` with a `unique_key` treats a re-arriving row as another version of a
row it already has, and overwrites in place. That is what makes re-landing
safe, and therefore what makes the lookback window above affordable.

In [12]:
assert shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_append") == 32
assert shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_append WHERE order_id = 12") == 2
assert shop.scalar("SELECT COUNT(*) FROM marts.fct_orders") == 31
assert shop.scalar("SELECT status FROM marts.fct_orders WHERE order_id = 12") == "shipped"
verdicts.append(("3 · row updated upstream", "append strategy", "silent", "yes",
                 "duplicate order; every aggregate now wrong"))
verdicts.append(("3 · row updated upstream", "merge + unique_key (committed)", "graceful", "—",
                 "row overwritten in place"))
print("✓ append: 32 rows with a duplicate · merge: 31 rows, order #12 shipped")

✓ append: 32 rows with a duplicate · merge: 31 rows, order #12 shipped


---

## Scenario 4 · A column appears upstream

A release adds a column to a feed nobody downstream asked for.

The committed staging model is immune by construction: it names its columns,
so a new one stops at the boundary until someone admits it deliberately. That
is a real and underrated firewall.

But incremental models written as `select *` are everywhere, and for those the
`on_schema_change` config decides what happens. There are four possible
answers, so rather than describe them, run all four, each on its own isolated
table, with the same event staged four times. One of the four fails loudly
below; that is the mode doing its job:

In [13]:
MODES = ["ignore", "append_new_columns", "sync_all_columns", "fail"]


def probe(mode: str) -> dict:
    """One throwaway raw table, one select-* incremental model. Add a column
    upstream between runs and record what this mode did about it."""
    table, model = f"probe_{mode}", f"inc_{mode}"
    shop.run_sql(f"DROP TABLE IF EXISTS raw.{table}")
    shop.run_sql(f"CREATE TABLE raw.{table} (id int, val int, loaded_at datetime2(0))")
    shop.run_sql(f"INSERT INTO raw.{table} VALUES (1, 10, '2026-07-01'), (2, 20, '2026-07-02')")

    scratch.model(model, f"""
{{{{ config(materialized='incremental', incremental_strategy='append',
            on_schema_change='{mode}', schema='scratch') }}}}
select * from warehouse.raw.{table}
{{% if is_incremental() %}}
where loaded_at > (select coalesce(max(loaded_at), '1900-01-01') from {{{{ this }}}})
{{% endif %}}
""")
    assert dbt("run", "--select", model, quiet=True).success  # first run: 2 rows

    shop.add_column(table, "extra int")                        # the release
    shop.run_sql(f"INSERT INTO raw.{table} VALUES (3, 30, '2026-07-03', 7)")
    rerun = dbt("run", "--select", model, quiet=True)

    cols = [r[0] for r in shop.run_sql(
        "SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS "
        "WHERE TABLE_SCHEMA = 'scratch' AND TABLE_NAME = ?", model)]
    return {
        "on_schema_change": mode,
        "second run": "OK" if rerun.success else "FAILED",
        "new column in mart": "extra" in cols,
        "rows": shop.scalar(f"SELECT COUNT(*) FROM scratch.{model}") if rerun.success else 2,
    }


results = pd.DataFrame([probe(m) for m in MODES])
results

12:42:31  1 of 1 ERROR creating sql incremental model scratch.inc_fail ................... [ERROR in 0.13s]


12:42:31  [ERROR]: in model inc_fail (models/scratch/inc_fail.sql)


12:42:31    Compilation Error in model inc_fail (models/scratch/inc_fail.sql)
  
                The source and target schemas on this incremental model are out of sync!
                They can be reconciled in several ways:
                  - set the `on_schema_change` config to either append_new_columns or sync_all_columns, depending on your situation.
                  - Re-run the incremental model with `full_refresh: True` to update the target schema.
                  - update the schema manually and re-run the process.
  
                Additional troubleshooting context:
                   Source columns not in target: [SQLServerColumnNative(column='extra', dtype='int', char_size=4, numeric_precision=10, numeric_scale=0)]
                   Target columns not in source: []
                   New column types: []
            
  
  > in macro default__process_schema_changes (macros/materializations/models/incremental/on_schema_change.sql)
  > called by macro process_schema_cha

,on_schema_change,second run,new column in mart,rows
0,ignore,OK,False,3
1,append_new_columns,OK,True,3
2,sync_all_columns,OK,True,3
3,fail,FAILED,False,2


### Verdict: the default drops data and says nothing

Read the table row by row.

`ignore` is the **default**. It loaded row 3 but threw away the value in its
new column; the mart never grew the column at all. Green run, narrower truth.

`append_new_columns` and `sync_all_columns` both altered the mart and kept the
data. The difference between them only shows up when a column is *removed*
(not staged here): sync drops it, append leaves it behind.

`fail` refused to run. The mart is stale but honest, and a person gets to
decide. Loud beats silent.

In [14]:
by_mode = results.set_index("on_schema_change")
assert by_mode.loc["ignore", "second run"] == "OK"
assert not by_mode.loc["ignore", "new column in mart"]
assert by_mode.loc["append_new_columns", "new column in mart"]
assert by_mode.loc["sync_all_columns", "new column in mart"]
assert by_mode.loc["fail", "second run"] == "FAILED"
verdicts.append(("4 · column added upstream", "on_schema_change: ignore (default)", "silent", "yes",
                 "new column's values discarded"))
verdicts.append(("4 · column added upstream", "append_new_columns / sync_all_columns", "graceful", "—",
                 "mart altered in place, data kept"))
verdicts.append(("4 · column added upstream", "on_schema_change: fail", "loud", "—",
                 "run refuses; mart stale but honest"))
print("✓ four modes, four outcomes · the default is the dangerous one")

✓ four modes, four outcomes · the default is the dangerous one


`fct_orders` sets `on_schema_change='fail'`, which you can see in its config
block above. It is the loud option, chosen because a mart that stops is
cheaper to deal with than a mart that quietly narrows. Its staging layer names
columns explicitly, so in practice drift never reaches it; the config is
insurance against the day someone changes that.

**Column *type* changes are harder** on SQL Server, because `ALTER COLUMN` has
real limits. This project opts into the adapter's `safe_type_expansion` flag
in [dbt_project.yml](../transform/dbt_project.yml), which permits only
provable widenings: `int` to `bigint`, and so on. Part 5 lands one of those
on purpose.

---

## Scenario 5 · Repairing the damage

The append variant still holds 32 rows with a duplicate. Incremental damage
does not heal on its own: the watermark has moved past it, so no future run
will revisit those rows.

The repair is `--full-refresh`, which rebuilds the table from source as though
it had never been incremental:

In [15]:
result = dbt("run", "--select", "fct_orders_append", "--full-refresh", quiet=True)
assert result.success
assert shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_append") == 31
assert shop.scalar("SELECT COUNT(*) FROM scratch.fct_orders_append WHERE order_id = 12") == 1
verdicts.append(("5 · repairing a damaged table", "--full-refresh", "graceful", "—",
                 "rebuilt from source, at full-rebuild price"))
print("✓ full refresh: 31 rows, duplicate gone")

✓ full refresh: 31 rows, duplicate gone


The same command is also how you prove the committed model is honest. A full
refresh recomputes `fct_orders` from scratch; if the incremental runs have
been correct all along, the rebuilt table must be **indistinguishable from the
one that was already there**.

That equivalence is the entire contract of an incremental model, and it is
worth asserting rather than assuming:

In [16]:
# Row count, money, and the newest timestamp: enough to catch a lost row, a
# duplicated one, or a stale merge. Not a byte comparison, a fingerprint.
fingerprint = "SELECT COUNT(*), SUM(line_total), MAX(updated_at) FROM marts.fct_orders"
before = shop.run_sql(fingerprint)[0]

result = dbt("run", "--select", "fct_orders", "--full-refresh", quiet=True)
assert result.success

after = shop.run_sql(fingerprint)[0]
assert tuple(before) == tuple(after), (before, after)
print(f"✓ incremental ≡ full rebuild: {before[0]} rows, ${before[1]}, unchanged")

✓ incremental ≡ full rebuild: 31 rows, $12234.75, unchanged


The catch is the price. This lab rebuilt 31 rows. Production rebuilds
billions: hours of warehouse time, with the mart degraded or locked while it
happens, and a bill at the end. `--full-refresh` is the fire extinguisher, not
the sprinkler system.

---

## What the scenarios add up to

Colour is risk to your data, not success of the run: green handled it, yellow
stopped and said so, red is a table that is now wrong. The quadrant to fear is
**silent + damage**, and the intuitive choice landed there three times: once
because it *is* dbt's default (`on_schema_change: ignore`), and twice because
it is simply what most people would write.

In [17]:
matrix = pd.DataFrame(
    verdicts, columns=["scenario", "configuration", "outcome", "data damage", "note"])
verdict.style(matrix)

scenario,configuration,outcome,data damage,note
"1 · re-run, nothing new",merge + unique_key (committed),graceful,—,"0 rows merged, count unchanged"
2 · order arrives late,watermark on event time,silent,yes,row is permanently below the mark
2 · order arrives late,watermark on arrival time (committed),graceful,—,caught on the next run; revenue corrected
3 · row updated upstream,append strategy,silent,yes,duplicate order; every aggregate now wrong
3 · row updated upstream,merge + unique_key (committed),graceful,—,row overwritten in place
4 · column added upstream,on_schema_change: ignore (default),silent,yes,new column's values discarded
4 · column added upstream,append_new_columns / sync_all_columns,graceful,—,"mart altered in place, data kept"
4 · column added upstream,on_schema_change: fail,loud,—,run refuses; mart stale but honest
5 · repairing a damaged table,--full-refresh,graceful,—,"rebuilt from source, at full-rebuild price"


In [18]:
silent_damage = matrix[(matrix["outcome"] == "silent") & (matrix["data damage"] == "yes")]
assert len(silent_damage) == 3
assert (matrix[matrix["configuration"].str.contains("committed")]["data damage"] == "—").all()
print("✓ 3 silent-damage rows · each one a default or an intuitive choice")
print("✓ the committed configuration survived every scenario")

✓ 3 silent-damage rows · each one a default or an intuitive choice
✓ the committed configuration survived every scenario


| the choice | why the committed model makes it |
|---|---|
| `merge`, not `append` | a re-arriving row means *update*, not *insert again* |
| `unique_key='order_id'` | gives merge its notion of identity: one order, one row |
| watermark on `loaded_at` | the loader's clock catches late events; the event's clock never can |
| explicit staging columns | schema drift stops at the boundary by construction |
| `on_schema_change` set deliberately | because the default is *ignore*, and ignore loses data |

**What about microbatch?** dbt 1.9 added an incremental strategy that takes
scenario 2 head-on: [microbatch](https://docs.getdbt.com/docs/build/incremental-microbatch)
filters on *event* time after all, but makes that survivable by rebuilding each
recent period as its own batch (the lookback insurance from scenario 2, built
into the strategy). The adapter here supports it, and the reasoning does not
change: an order that arrives after the lookback window has already moved past
it is lost exactly the way order #31 was.

One loss stayed out of scope here. Scenario 3's merge kept the mart *current*,
not *complete*: order #12 now reads `shipped`, but its `placed` version is gone
from `raw` forever, and no incremental strategy can bring it back. A row
*deleted* upstream is worse still: it never re-lands, so no incremental run
ever hears about it.

Keeping history that the source itself discards requires a different tool.

*Next: [when the source forgets →](03-snapshots-and-history.ipynb)*